# **Machine Learning Project PART 2 | Step 1 - ATU Winter 2024**  
**Author**: Lais Coletta Pereira  
**Lecturer**: Brian McGinley  

---

## Data Pre-processing and management:
The first task will be to build a dataset from the annotated data:

In [1]:
#Imports
import os
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy import signal

In [2]:
# Define output directory for spectrogram files and create it if not exists
output_dir = 'spectrogram_output'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

##### Function to load audio and annotations, reads .wav and .txt annotation files:

In [3]:
def load_audio_and_annotation(file_base_name):
    # Load the WAV audio file
    audio_path = file_base_name + '.wav'
    sample_rate, samples = wavfile.read(audio_path)
    
    # Load the annotation file (typically .txt or .csv)
    annot_file_path = file_base_name + '.Table.1.selections.txt'
    df = pd.read_csv(annot_file_path, sep='\t')
    
    return sample_rate, samples, df

**Define a function to extract audio segments based on annotations**

Extract audio segments based on start and end times from annotations, storing each segment along with its relevant information in a list.

In [4]:
def extract_audio_segments(samples, sample_rate, annotations):
    segments = []
    
    # Iterate through each row of the annotation DataFrame and extract the audio
    for _, row in annotations.iterrows():
        start_time = row['Begin Time (s)']
        end_time = row['End Time (s)']
        
        # Convert start and end time to sample indices based on sample_rate
        start_sample = int(start_time * sample_rate)
        end_sample = int(end_time * sample_rate)
        
        # Extract the corresponding audio segment
        segment = samples[start_sample:end_sample]
        
        # Append the segment info to the list of segments
        segments.append({
            'Selection': row['Selection'],
            'View': row['View'],
            'Channel': row['Channel'],
            'Begin Time (s)': start_time,
            'End Time (s)': end_time,
            'Low Freq (Hz)': row['Low Freq (Hz)'],
            'High Freq (Hz)': row['High Freq (Hz)'],
            'Delta Time (s)': row['Delta Time (s)'],
            'Delta Freq (Hz)': row['Delta Freq (Hz)'],
            'Avg Power Density (dB FS/Hz)': row['Avg Power Density (dB FS/Hz)'],
            'Annotation': row['Annotation'],
            'audio_segment': segment  # Extracted audio segment
        })
        
    return segments


**Define a function to generate a spectrogram from the audio segment**

This function generates a spectrogram from an audio segment by applying the Short-Time Fourier Transform (STFT) using [scipy.signal.spectrogram](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.spectrogram.html). 
It allows us to specify the frequency range (fmin, fmax) and other parameters like window length, FFT points, and overlap to control the resolution of the spectrogram. The function returns frequency bins, time bins, and the spectrogram data (2D array). I am using the spectrogram.ipynp as a source for this part.


In [5]:
# Reference code https://stackoverflow.com/questions/38015319/how-to-create-a-numpy-array-from-a-pydub-audiosegment
def generate_spectrogram(audio_segment, sample_rate, fmin=20, fmax=1000, nperseg=2456, nfft=4096, noverlap=1228):
    """
    Generate spectrogram for a given audio segment based on frequency range (fmin, fmax).
    
    Parameters:
        - audio_segment (np.array): The raw audio data.
        - sample_rate (int): The sampling rate of the audio data.
        - fmin (int): The minimum frequency for the spectrogram. Default is 20 Hz.
        - fmax (int): The maximum frequency for the spectrogram. Default is 1000 Hz.
        - nperseg (int): The length of each segment for the FFT. Default is 2456 samples.
        - nfft (int): The number of FFT points. Default is 4096.
        - noverlap (int): The number of points to overlap between segments. Default is 1228.

    Returns:
        - frequencies (np.array): Frequency bins for the spectrogram.
        - times (np.array): Time bins for the spectrogram.
        - spectrogram_data (np.array): 2D spectrogram array.
    """
    # Generate the spectrogram using scipy.signal.spectrogram
    frequencies, times, spectrogram_data = signal.spectrogram(audio_segment, sample_rate, 
                                                              nperseg=nperseg, nfft=nfft, 
                                                              noverlap=noverlap, window='hann')
    
    # Threshold tiny values in the spectrogram (to avoid plotting issues) 
    spectrogram_data[spectrogram_data < 0.001] = 0.001
    
    # Slice the frequencies based on the desired range (fmin to fmax)
    freq_slice = np.where((frequencies >= fmin) & (frequencies <= fmax))
    frequencies = frequencies[freq_slice]
    spectrogram_data = spectrogram_data[freq_slice, :]
    
    return frequencies, times, spectrogram_data

Process audio files and their annotations, extract corresponding segments based on the annotations, and combines them into a dataset:

In [6]:
# Define folder paths where the .wav and annotation files are stored
folder_paths = [
    r'C:\Users\Admin\Downloads\Samples Grey Seal\Guttural rupe',
    r'C:\Users\Admin\Downloads\Samples Grey Seal\Rupes A and B',
    r'C:\Users\Admin\Downloads\Samples Grey Seal\Moan'
]

# Initialize a list to collect DataFrames of audio segments
dataset = []

# Iterate over each folder and process audio files and their annotations
for folder_path in folder_paths:
    for file in os.listdir(folder_path):
        if file.endswith('.wav'):
            # Get the base file name without extension
            file_base_name = os.path.splitext(file)[0]
            file_base_path = os.path.join(folder_path, file_base_name)
            
            # Load audio and its corresponding annotations
            sample_rate, samples, df = load_audio_and_annotation(file_base_path)
            
            # Extract the audio segments based on annotation timestamps
            segments = extract_audio_segments(samples, sample_rate, df)
            
            # Create a DataFrame from the extracted segments
            df_segments = pd.DataFrame(segments)
            
            # Add this DataFrame to the larger dataset
            dataset.append(df_segments)

# Concatenate all the segment DataFrames into one final dataset
final_df = pd.concat(dataset, ignore_index=True)

# Print the first few rows of the combined dataset for validation
print(final_df.head())

# Process each segment to generate a spectrogram and save as an .npz file
for index, segment_info in final_df.iterrows():
    # Extract relevant segment information
    audio_segment = segment_info['audio_segment']
    start_time = segment_info['Begin Time (s)']
    end_time = segment_info['End Time (s)']
    
    # Frequency bounds based on annotations for each segment
    low_freq = segment_info['Low Freq (Hz)']
    high_freq = segment_info['High Freq (Hz)']
    
    # Generate the spectrogram for the segment using dynamic frequency bounds
    frequencies, times, spectrogram_data = generate_spectrogram(audio_segment, sample_rate, 
                                                               fmin=low_freq, fmax=high_freq)
    
    # Ensure the output directory exists
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Prepare metadata about the segment to save alongside the spectrogram
    metadata = {
        'Selection': segment_info['Selection'],
        'Begin Time (s)': start_time,
        'End Time (s)': end_time,
        'Annotation': segment_info['Annotation'],
        'Spectrogram Shape': spectrogram_data.shape
    }
    
    # Construct the filename for saving the spectrogram data and metadata
    spectrogram_filename = f"{output_dir}/{segment_info['Selection']}_{start_time}-{end_time}_spectrogram.npz"
    
    # Save the spectrogram data and metadata in an .npz file
    np.savez(spectrogram_filename, spectrogram=spectrogram_data, metadata=metadata)

   Selection           View  Channel  Begin Time (s)  End Time (s)  \
0          1  Spectrogram 1        1        4.136987      4.657535   
1          2  Spectrogram 1        1        3.789955      4.264841   
2          3  Spectrogram 1        1       14.840188     15.406398   
3          4  Spectrogram 1        1       24.543802     25.082615   
4          5  Spectrogram 1        1       35.989220     36.573694   

   Low Freq (Hz)  High Freq (Hz)  Delta Time (s)  Delta Freq (Hz)  \
0        173.956         286.792          0.5205          112.836   
1        371.419         451.345          0.4749           79.926   
2        136.660         445.248          0.5662          308.588   
3        167.519         392.347          0.5388          224.828   
4        119.027         436.431          0.5845          317.405   

   Avg Power Density (dB FS/Hz) Annotation  \
0                        -82.85     G rupe   
1                        -92.16     G rupe   
2                        -

C:\Users\Admin\AppData\Local\Temp\ipykernel_13604\3761410645.py:21: UserWarning: nperseg = 2456 is greater than input length  = 2192, using nperseg = 2192
  frequencies, times, spectrogram_data = signal.spectrogram(audio_segment, sample_rate,


## Calculate the Spectogram

_My advice would be to extract a spectrogram for each call. Each spectrogram should be the
same size so there will be some pre-pre-processing to find what is the longest call (in time) and
the broadest in frequency. This will serve as the baseline for the largest spectrogram. Extract
and save (with the metadata)a spectrogram for each call (you can calculate the central time of
each call from the metadata). Note, don’t save as images but as a raw 2d array of numbers.
You can also create spectrograms for the extra class “no-call” where you build a set of
spectrograms from times when there is no annotated call. You may assume that any
unannotated region has no call in it. Take care to ensure that the extracted“no-call”
spectrograms are from the same frequency region as the call spectrograms.
I would recommend having a single jupyter notebook file that does all this preprocessing._

For each file and segment, the code generates a spectrogram using generate_spectrogram(). The spectrogram and metadata are stored in the respective lists for later use.

### Bibliography

1. McKinney, W. (2010). Data Structures for Statistical Computing in Python. *Proceedings of the 9th Python in Science Conference*, 51-56. https://doi.org/10.25080/Majora-92bf1922-00

2. Virtanen, P., Gommers, R., Oliphant, T., et al. (2020). SciPy 1.0: Fundamental Algorithms for Scientific Computing in Python. *Nature Methods*, 17(3), 261-272. https://doi.org/10.1038/s41592-019-0686-2

3. Brown, J.C. (1992). Computational Analysis of Sound Patterns. *Journal of the Acoustical Society of America*, 92(5), 2795-2798. https://doi.org/10.1121/1.407930

4. Oppenheim, A.V., Schafer, R.W. (1989). *Discrete-Time Signal Processing*. Prentice Hall.

https://stackoverflow.com/questions/44787437/how-to-convert-a-wav-file-to-a-spectrogram-in-python3

https://stackoverflow.com/questions/38015319/how-to-create-a-numpy-array-from-a-pydub-audiosegment
